# 01 — Embeddings & Semantic Search

This notebook explores the foundational building block of any RAG pipeline: **text embeddings**.

We'll learn:
- What embeddings look like (vectors of numbers)
- How cosine similarity measures semantic closeness
- How to build a mini semantic search engine in ~10 lines of code

**Model used:** `all-MiniLM-L6-v2` — a fast, open-source embedding model (384 dimensions).

In [ ]:
from sentence_transformers import SentenceTransformer, util
import numpy as np

# Load embedding model (~80MB download on first run)
model = SentenceTransformer("all-MiniLM-L6-v2")
print(f"Model loaded. Embedding dimension: {model.get_sentence_embedding_dimension()}")

## What does an embedding look like?

An embedding is just a list of numbers — a **vector** — that captures the *meaning* of a piece of text.

In [ ]:
text = "The cat sat on the mat"
embedding = model.encode(text)

print(f"Type: {type(embedding)}")
print(f"Shape: {embedding.shape}")
print(f"First 10 values: {embedding[:10].round(4)}")
print(f"Min: {embedding.min():.4f}, Max: {embedding.max():.4f}")

## Cosine similarity: measuring meaning

Similar meaning → nearby vectors → high cosine similarity (close to 1.0).

In [ ]:
pairs = [
    ("How do I reset my password?", "I forgot my login credentials"),
    ("How do I reset my password?", "The weather is nice today"),
    ("The dog ran quickly", "A canine sprinted fast"),
    ("Quantum physics research", "Company vacation policy"),
]

print(f"{'Sentence A':<35} {'Sentence B':<35} {'Similarity':>10}")
print("-" * 82)

for a, b in pairs:
    emb_a = model.encode(a)
    emb_b = model.encode(b)
    sim = util.cos_sim(emb_a, emb_b).item()
    print(f"{a:<35} {b:<35} {sim:>10.4f}")

## Mini semantic search engine

This is the **retrieval core of RAG** — in ~10 lines of code.

In [ ]:
# Our "knowledge base" — imagine these are chunks from company documents
documents = [
    "The company offers 12 weeks of paid parental leave.",
    "Employees can expense up to $500 per year for learning and development.",
    "Remote work is allowed up to 3 days per week.",
    "Health insurance covers dental and vision.",
    "The office is closed on all federal holidays.",
    "Annual performance reviews happen in Q4.",
    "The 401(k) plan matches up to 4% of salary.",
    "Unlimited PTO is available after the first year of employment.",
]

# Embed all documents (done once, ahead of time in a real system)
doc_embeddings = model.encode(documents)

In [ ]:
def search(query: str, top_k: int = 3):
    """Search documents by semantic similarity."""
    query_embedding = model.encode(query)
    similarities = util.cos_sim(query_embedding, doc_embeddings)[0]

    top_indices = similarities.argsort(descending=True)[:top_k]

    print(f"Query: '{query}'\n")
    for rank, idx in enumerate(top_indices, 1):
        print(f"  {rank}. [{similarities[idx]:.3f}] {documents[idx]}")
    print()


# Try different queries
search("Can I work from home?")
search("What happens when I have a baby?")
search("How much vacation do I get?")
search("Does the company help with retirement savings?")

## Using our package module

The same logic, but using the `rag_pipeline.embeddings` module we'll build up over the course:

In [ ]:
import sys
sys.path.insert(0, "../src")

from rag_pipeline.embeddings import load_model, rank_by_similarity

model = load_model()
results = rank_by_similarity("Can I work from home?", documents, model, top_k=3)

for doc, score in results:
    print(f"[{score:.3f}] {doc}")

## Key takeaways

1. **Embeddings** convert text meaning into vectors — numbers you can do math on
2. **Cosine similarity** tells you how close two meanings are (0 = unrelated, 1 = identical)
3. **Semantic search** beats keyword search because it understands meaning, not just words
4. This is the **retrieval engine of RAG** — the "R" in the name

**Next notebook:** Document chunking — how to split real PDFs into pieces that embed well.